In [ ]:
print("Har Har Mahadev")

In [6]:
"""
preprocess.py — HAV-DF Dataset Preprocessing
Extracts:
  - Face crops from video frames (via dlib or MTCNN fallback)
  - Mel-spectrograms from audio tracks (via librosa)

Usage:
    python preprocess.py --data_root /path/to/HAV-DF --output_dir ./processed
"""

import os
import cv2
import json
import argparse
import numpy as np
import librosa
import librosa.display
from pathlib import Path
from tqdm import tqdm
import subprocess

# ── Optional face detection backends ─────────────────────────────────────────
try:
    import dlib
    DLIB_AVAILABLE = True
except ImportError:
    DLIB_AVAILABLE = False

try:
    from facenet_pytorch import MTCNN
    import torch
    MTCNN_AVAILABLE = True
except ImportError:
    MTCNN_AVAILABLE = False

# ── Config ─────────────────────────────────────────────────────────────────────
FRAME_RATE       = 1        # frames per second to sample (1 fps is enough for CPU)
FACE_SIZE        = 224      # face crop resolution (H x W)
N_MELS           = 128      # mel-spectrogram bins
HOP_LENGTH       = 512
N_FFT            = 1024
AUDIO_SR         = 16000    # resample all audio to 16 kHz
MAX_AUDIO_SEC    = 10       # truncate/pad audio to this length
MAX_FRAMES       = 10       # max face frames per video


# ── Face detector factory ──────────────────────────────────────────────────────
def get_face_detector():
    if DLIB_AVAILABLE:
        detector = dlib.get_frontal_face_detector()
        print("[INFO] Using dlib face detector.")
        return ("dlib", detector)
    elif MTCNN_AVAILABLE:
        detector = MTCNN(keep_all=False, device="cpu")
        print("[INFO] Using MTCNN face detector.")
        return ("mtcnn", detector)
    else:
        print("[WARN] No face detector found. Using full-frame fallback.")
        return ("none", None)


def detect_face_dlib(frame_rgb, detector):
    dets = detector(frame_rgb, 1)
    if len(dets) == 0:
        return None
    d = dets[0]
    return (d.left(), d.top(), d.right(), d.bottom())


def detect_face_mtcnn(frame_rgb, detector):
    boxes, _ = detector.detect(frame_rgb)
    if boxes is None or len(boxes) == 0:
        return None
    b = boxes[0].astype(int)
    return (b[0], b[1], b[2], b[3])


def crop_face(frame_bgr, bbox, size=FACE_SIZE, margin=0.25):
    """Crop face with margin, resize to size x size."""
    h, w = frame_bgr.shape[:2]
    x1, y1, x2, y2 = bbox
    bw, bh = x2 - x1, y2 - y1
    mx, my = int(bw * margin), int(bh * margin)
    x1 = max(0, x1 - mx)
    y1 = max(0, y1 - my)
    x2 = min(w, x2 + mx)
    y2 = min(h, y2 + my)
    face = frame_bgr[y1:y2, x1:x2]
    if face.size == 0:
        return None
    return cv2.resize(face, (size, size))


# ── Audio extraction ───────────────────────────────────────────────────────────
def extract_audio_from_video(video_path: str, out_wav: str, sr: int = AUDIO_SR):
    """Use ffmpeg to extract audio from video."""
    cmd = [
        "ffmpeg", "-y", "-i", video_path,
        "-ac", "1",               # mono
        "-ar", str(sr),            # resample
        "-vn",                     # no video
        out_wav
    ]
    result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return result.returncode == 0


def wav_to_melspectrogram(wav_path: str, sr: int = AUDIO_SR,
                           n_mels: int = N_MELS, max_sec: int = MAX_AUDIO_SEC):
    """
    Returns mel-spectrogram as float32 numpy array [n_mels, T].
    Truncated or zero-padded to max_sec * sr samples.
    """
    y, _ = librosa.load(wav_path, sr=sr, mono=True)
    max_samples = max_sec * sr
    if len(y) >= max_samples:
        y = y[:max_samples]
    else:
        y = np.pad(y, (0, max_samples - len(y)))

    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=n_mels
    )
    mel_db = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
    # Normalize to [-1, 1]
    mel_db = mel_db / 80.0  # librosa dB range is typically [-80, 0]
    return mel_db  # [n_mels, T]


# ── Video frame extraction ─────────────────────────────────────────────────────
def extract_face_frames(video_path: str, detector_info, fps: int = FRAME_RATE,
                         max_frames: int = MAX_FRAMES):
    """
    Extracts up to max_frames face crops from video at given fps.
    Returns list of numpy arrays [H, W, 3] (BGR, uint8).
    """
    dtype, detector = detector_info
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    native_fps = cap.get(cv2.CAP_PROP_FPS) or 25
    frame_interval = max(1, int(native_fps / fps))

    faces = []
    frame_idx = 0
    while len(faces) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            bbox = None
            if dtype == "dlib":
                bbox = detect_face_dlib(frame_rgb, detector)
            elif dtype == "mtcnn":
                bbox = detect_face_mtcnn(frame_rgb, detector)

            if bbox is not None:
                face = crop_face(frame, bbox)
            else:
                # fallback: center crop
                h, w = frame.shape[:2]
                s = min(h, w)
                y0, x0 = (h - s) // 2, (w - s) // 2
                face = cv2.resize(frame[y0:y0+s, x0:x0+s], (FACE_SIZE, FACE_SIZE))

            if face is not None:
                faces.append(face)
        frame_idx += 1

    cap.release()
    return faces


# ── Main processing ────────────────────────────────────────────────────────────
def process_dataset(data_root: str, output_dir: str):
    """
    Expected dataset structure (adjust as needed for actual HAV-DF layout):
      data_root/
        real/   *.mp4
        fake/   *.mp4

    Outputs:
      output_dir/
        real/<video_id>/
          frame_000.npy  ... face crops [224, 224, 3]
          mel.npy        ... mel-spectrogram [128, T]
        fake/<video_id>/
          ...
        manifest.json   ... list of {id, label, n_frames, has_audio}
    """
    data_root  = Path(data_root)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    detector_info = get_face_detector()
    manifest = []
    tmp_wav = "/tmp/havdf_tmp_audio.wav"

    for label_name, label_int in [("real", 0), ("fake", 1)]:
        src_dir = data_root / label_name
        if not src_dir.exists():
            print(f"[WARN] {src_dir} not found, skipping.")
            continue

        videos = sorted(src_dir.glob("*.mp4")) + sorted(src_dir.glob("*.avi"))
        print(f"\n[INFO] Processing {len(videos)} {label_name} videos...")

        for vpath in tqdm(videos, desc=label_name):
            vid_id = vpath.stem
            out_vid_dir = output_dir / label_name / vid_id
            out_vid_dir.mkdir(parents=True, exist_ok=True)

            # ── Face frames ────────────────────────────────────────────────
            frames_done = False
            n_frames    = 0
            frame_npys  = list(out_vid_dir.glob("frame_*.npy"))
            if frame_npys:
                n_frames    = len(frame_npys)
                frames_done = True
            else:
                faces = extract_face_frames(str(vpath), detector_info)
                for i, face in enumerate(faces):
                    np.save(out_vid_dir / f"frame_{i:03d}.npy", face)
                n_frames    = len(faces)
                frames_done = n_frames > 0

            # ── Audio mel-spec ─────────────────────────────────────────────
            has_audio  = False
            mel_path   = out_vid_dir / "mel.npy"
            if mel_path.exists():
                has_audio = True
            else:
                ok = extract_audio_from_video(str(vpath), tmp_wav)
                if ok and os.path.exists(tmp_wav):
                    try:
                        mel = wav_to_melspectrogram(tmp_wav)
                        np.save(mel_path, mel)
                        has_audio = True
                    except Exception as e:
                        print(f"[WARN] Audio mel failed for {vid_id}: {e}")

            manifest.append({
                "id":        vid_id,
                "label":     label_int,
                "label_str": label_name,
                "n_frames":  n_frames,
                "has_audio": has_audio,
                "path":      str(out_vid_dir.relative_to(output_dir))
            })

    # Save manifest
    manifest_path = output_dir / "manifest.json"
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    total  = len(manifest)
    n_real = sum(1 for m in manifest if m["label"] == 0)
    n_fake = sum(1 for m in manifest if m["label"] == 1)
    print(f"\n[DONE] Processed {total} videos: {n_real} real, {n_fake} fake.")
    print(f"       Manifest saved to {manifest_path}")


parser = argparse.ArgumentParser()
parser.add_argument("--data_root", required=True, help="Path to HAV-DF dataset root")
parser.add_argument("--output_dir", default="./processed", help="Path to output directory")
args = parser.parse_args([
    "--data_root", "D:/projects/project_pashupatastra/data/audio_video_deepfake/HAV-DF",
    "--output_dir", "D:/projects/project_pashupatastra/data/audio_video_deepfake"
])
process_dataset(args.data_root, args.output_dir)

[WARN] No face detector found. Using full-frame fallback.
[WARN] D:\projects\project_pashupatastra\data\audio_video_deepfake\HAV-DF\real not found, skipping.
[WARN] D:\projects\project_pashupatastra\data\audio_video_deepfake\HAV-DF\fake not found, skipping.

[DONE] Processed 0 videos: 0 real, 0 fake.
       Manifest saved to D:\projects\project_pashupatastra\data\audio_video_deepfake\manifest.json


In [8]:
"""
dataset.py — PyTorch Dataset for HAV-DF processed outputs.

Each sample returns:
  frames : Tensor [MAX_FRAMES, 3, 224, 224]  float32, normalized
  mel    : Tensor [1, N_MELS, T]             float32
  label  : Tensor scalar                     int64 (0=real, 1=fake)
"""

import json
import random
import numpy as np
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T

# ── Constants matching preprocess.py ──────────────────────────────────────────
MAX_FRAMES = 10
FACE_SIZE  = 224
N_MELS     = 128
MEL_T      = 313   # time frames for 10 sec audio @ sr=16000, hop=512

# ── ImageNet normalization for the video backbone ──────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def build_frame_transform(augment: bool = False):
    ops = []
    if augment:
        ops += [
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
            T.RandomGrayscale(p=0.05),
            T.RandomRotation(degrees=10),
            T.RandomResizedCrop(FACE_SIZE, scale=(0.85, 1.0)),
        ]
    else:
        ops += [T.CenterCrop(FACE_SIZE)]
    ops += [
        T.ConvertImageDtype(torch.float32),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
    return T.Compose(ops)


class HAVDFDataset(Dataset):
    """
    Loads preprocessed HAV-DF samples from manifest.json.

    Args:
        processed_dir : path to output_dir from preprocess.py
        split         : 'train' | 'val' | 'test' (uses pre-split manifest keys)
        augment       : apply video augmentation (train only)
        manifest_path : override default manifest.json location
    """

    def __init__(self, processed_dir: str, entries: list,
                 augment: bool = False):
        self.root      = Path(processed_dir)
        self.entries   = entries
        self.augment   = augment
        self.frame_tfm = build_frame_transform(augment)

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]
        vid_dir = self.root / entry["path"]

        # ── Load face frames ───────────────────────────────────────────────
        frame_paths = sorted(vid_dir.glob("frame_*.npy"))
        frames = []
        for fp in frame_paths[:MAX_FRAMES]:
            arr = np.load(fp)           # [H, W, 3] BGR uint8
            arr = arr[:, :, ::-1].copy() # BGR → RGB
            t = torch.from_numpy(arr).permute(2, 0, 1)   # [3, H, W] uint8
            t = self.frame_tfm(t)        # [3, 224, 224] float32
            frames.append(t)

        # Pad to MAX_FRAMES with zeros if needed
        while len(frames) < MAX_FRAMES:
            frames.append(torch.zeros(3, FACE_SIZE, FACE_SIZE))

        frames = torch.stack(frames)  # [MAX_FRAMES, 3, 224, 224]

        # ── Load mel-spectrogram ───────────────────────────────────────────
        mel_path = vid_dir / "mel.npy"
        if mel_path.exists():
            mel = np.load(mel_path).astype(np.float32)   # [N_MELS, T]
            # Pad or truncate T dimension
            if mel.shape[1] < MEL_T:
                mel = np.pad(mel, ((0, 0), (0, MEL_T - mel.shape[1])))
            else:
                mel = mel[:, :MEL_T]
            if self.augment:
                mel = self._augment_mel(mel)
            mel_tensor = torch.from_numpy(mel).unsqueeze(0)  # [1, N_MELS, MEL_T]
        else:
            mel_tensor = torch.zeros(1, N_MELS, MEL_T)

        label = torch.tensor(entry["label"], dtype=torch.long)
        return frames, mel_tensor, label

    # ── Mel augmentation ──────────────────────────────────────────────────────
    @staticmethod
    def _augment_mel(mel: np.ndarray) -> np.ndarray:
        """SpecAugment-lite: random frequency & time masking."""
        mel = mel.copy()
        n_mels, T = mel.shape

        # Frequency masking (mask up to 15 mel bins)
        f = random.randint(0, 15)
        f0 = random.randint(0, max(0, n_mels - f - 1))
        mel[f0:f0 + f, :] = mel.min()

        # Time masking (mask up to 30 time frames)
        t = random.randint(0, 30)
        t0 = random.randint(0, max(0, T - t - 1))
        mel[:, t0:t0 + t] = mel.min()

        return mel


# ── Train / Val / Test split utility ──────────────────────────────────────────
def load_splits(processed_dir: str, val_ratio: float = 0.15,
                test_ratio: float = 0.15, seed: int = 42):
    """
    Loads manifest.json and returns (train_entries, val_entries, test_entries).
    Stratified split by label.
    """
    manifest_path = Path(processed_dir) / "manifest.json"
    with open(manifest_path) as f:
        manifest = json.load(f)

    rng = random.Random(seed)

    real_entries = [m for m in manifest if m["label"] == 0]
    fake_entries = [m for m in manifest if m["label"] == 1]

    def split_list(lst):
        lst = lst.copy()
        rng.shuffle(lst)
        n      = len(lst)
        n_test = int(n * test_ratio)
        n_val  = int(n * val_ratio)
        return lst[n_test + n_val:], lst[n_test:n_test + n_val], lst[:n_test]

    real_tr, real_val, real_te = split_list(real_entries)
    fake_tr, fake_val, fake_te = split_list(fake_entries)

    train = real_tr + fake_tr
    val   = real_val + fake_val
    test  = real_te + fake_te

    rng.shuffle(train)
    print(f"[INFO] Split — train: {len(train)}, val: {len(val)}, test: {len(test)}")
    return train, val, test


def build_dataloaders(processed_dir: str, batch_size: int = 8,
                      num_workers: int = 0, seed: int = 42):
    """
    Returns (train_loader, val_loader, test_loader).
    Uses WeightedRandomSampler for class imbalance (308 fake vs 200 real).
    """
    train_entries, val_entries, test_entries = load_splits(processed_dir, seed=seed)

    train_ds = HAVDFDataset(processed_dir, train_entries, augment=True)
    val_ds   = HAVDFDataset(processed_dir, val_entries,   augment=False)
    test_ds  = HAVDFDataset(processed_dir, test_entries,  augment=False)

    # Weighted sampler for class balance
    labels     = [e["label"] for e in train_entries]
    class_counts = [labels.count(0), labels.count(1)]
    weights    = [1.0 / class_counts[l] for l in labels]
    sampler    = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                               num_workers=num_workers, pin_memory=False)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                               num_workers=num_workers, pin_memory=False)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                               num_workers=num_workers, pin_memory=False)

    return train_loader, val_loader, test_loader



import sys
# Instead of sys.argv, use the actual path where your processed data is stored
processed_dir = "D:/projects/project_pashupatastra/data/audio_video_deepfake" 

# Verify the manifest exists before proceeding
import os
manifest_check = os.path.join(processed_dir, "manifest.json")
if not os.path.exists(manifest_check):
    print(f"ERROR: Could not find manifest at {manifest_check}. Check your path!")
else:
    # Build loaders
    tr, va, te = build_dataloaders(processed_dir, batch_size=4)
    
    # Get a sample batch
    frames, mel, labels = next(iter(tr))
    print(f"frames: {frames.shape}, mel: {mel.shape}, labels: {labels}")

[INFO] Split — train: 0, val: 0, test: 0


ValueError: num_samples should be a positive integer value, but got num_samples=0